# Visium HD analysis with Spartan: developing human esophagus and stomach

This notebook reproduces and explains the Visium HD analysis used to generate the main-paper Figure 5 and Figure 6 panels for the developing human esophagus and stomach sample at approximately 10 weeks gestation. The analysis uses the `SpatialData` framework to keep the high-resolution tissue geometry, H&E image, and associated `AnnData` table aligned throughout the workflow.

The notebook is organized around three biological questions:

1. **Can Spartan recover coherent high-resolution tissue domains?**  
   We apply Spartan to the Visium HD table and visualize the resulting domains directly in tissue space.

2. **Can Spartan resolve the gastroesophageal junction (GEJ)?**  
   We focus on the GEJ-associated domain, perform subclustering within this domain, and evaluate marker signatures distinguishing epithelial and mesenchymal subregions.

3. **Can Spartan reveal functional stromal organization?**  
   We examine representative domain-associated genes and module-level programs across stromal domains, including ECM, glycosaminoglycan/sulfation, and vascular/stromal signatures.

All clustering steps are unsupervised. Marker-weight analyses, module scores, and pathway summaries are used downstream to interpret the biological identity of the Spartan-derived domains and subdomains.

## 0. Define input data path

The analysis starts from a pre-processed Visium HD `SpatialData` object stored in Zarr format. This object contains the tissue image, spatial shapes/geometries, and the filtered expression table used for Spartan analysis.

Update this path if the dataset is stored in a different location on your machine.

In [ ]:
visium_hd_zarr_path = "/PATH_TO/RJ4_D1_Andy_Esophagus_processed_clusters_with_raw_counts.zarr"

## 1. Import packages and configure plotting

This cell imports the scientific Python stack, the Spartan package, `SpatialData` utilities, plotting libraries, Leiden clustering, and marker-classification utilities used throughout the notebook. A fixed seed is defined for reproducible stochastic steps such as PCA and Leiden clustering. Plotting parameters are configured for high-resolution output suitable for main-figure and supplementary-figure generation.

In [ ]:
import scanpy as sc
import spartan as sp
import pandas as pd
import numpy as np
import spatialdata as sd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl  
import leidenalg
import spatialdata_io
import spatialdata_plot
from sklearn.metrics import adjusted_rand_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
seed = 1

import IPython.display
from matplotlib_inline.backend_inline import set_matplotlib_formats
IPython.display.set_matplotlib_formats = set_matplotlib_formats

sc.settings.verbosity = 0
sc.settings.set_figure_params(
    dpi=600,
    dpi_save=600,
    facecolor="white",
    frameon=False,
)

## 2. Load the SpatialData object

Here we load the Visium HD dataset as a `SpatialData` object. This preserves the relationship between the expression table, spatial coordinates, tissue geometry, and high-resolution H&E image. The subsequent Spartan domains are written back into the associated `AnnData` table so they can be visualized directly on the tissue geometry.

In [ ]:
sdata = sd.read_zarr(visium_hd_zarr_path)

### Make gene names unique

Spatial transcriptomics objects can occasionally contain duplicated gene symbols. Making `var_names` unique avoids downstream ambiguity when subsetting genes, computing marker expression, or plotting gene-level signals.

In [ ]:
for table in sdata.tables.values():
    table.var_names_make_unique()

### Inspect the SpatialData container

This inspection step confirms the available images, shapes, coordinate systems, and tables stored inside the `SpatialData` object. For reviewer-facing reproducibility, this is useful because it makes the spatial organization of the object explicit before analysis.

In [ ]:
sdata

### Inspect the Visium HD expression table

The table `manual_analysis_count_area_filtered` is the main `AnnData` object used for domain identification. It contains filtered Visium HD spots/bins, expression features, spatial coordinates, and associated metadata.

In [ ]:
sdata['manual_analysis_count_area_filtered']

## 3. Highly variable gene annotation

Highly variable genes are annotated using the Seurat-style procedure. In this notebook, this step is primarily used to characterize the expression table and support downstream interpretation. Spartan itself uses PCA-derived expression structure together with spatial and local activation graphs to construct domains.

In [ ]:
sc.pp.highly_variable_genes(sdata['manual_analysis_count_area_filtered'], flavor='seurat', n_top_genes=2000)
sdata['manual_analysis_count_area_filtered']

## 4. Visualize the H&E image and analysis geometry

Before running Spartan, we visualize the H&E image together with the analysis geometry. This confirms that the expression table and spatial shapes are correctly aligned to the tissue coordinate system.

In [ ]:
plt.rcParams['axes.grid'] = False 
sdata.pl.render_images("RJ4_D1_Andy_Esophagus_hires_image").pl.render_shapes(
    "analysis_geometry", color=None,method="matplotlib", fill_alpha=0.5,table_name="manual_analysis_count_area_filtered"
).pl.show(coordinate_systems="RJ4_D1_Andy_Esophagus",figsize=(10, 10),title = "H&E image",frameon=False, dpi=600)

## 5. Configure Spartan for high-resolution Visium HD

This cell defines the Spartan parameters used for the Visium HD domain analysis. Because Visium HD has grid-like spatial organization, the spatial graph is constructed using `spatial_coord='grid'`. The KNN neighborhood and ring settings define the local spatial topology used to build the spatial graph and Local Spatial Activation (LSA) graph.

The three graph components are combined as:


`J` = ($\alpha$ - $\beta_1$)`L` + (1 - $\alpha$)`G` + ($\alpha$ - $\beta_2$)`S`,


where `L` is the Local Spatial Activation graph, `G` is the gene expression connectivity graph, and `S` is the spatial topology graph. The resulting aggregated graph `J` is clustered with Leiden.

### Rationale for the fixed graph-weighting setting used in single-cell imaging analyses

For the single-cell imaging-based analyses, we used a fixed graph-weighting setting of:

$
\alpha = 0.80,\quad \beta_1 = 0.10,\quad \beta_2 = 0.40.
$

Under the selected setting, the effective graph contributions are:

$
(\alpha-\beta_1)=0.70,\quad (1-\alpha)=0.20,\quad (\alpha-\beta_2)=0.40.
$

Thus, the LSA graph receives the strongest contribution, followed by the spatial adjacency graph and the gene-expression connectivity graph.

This weighting is motivated by both empirical and conceptual considerations. Empirically, single-cell imaging-based datasets such as MERFISH and Vizgen MERFISH exhibited stable high-performing operating regimes at relatively high \($\alpha$\) values, typically within the range \($\alpha$ $\approx$ 0.70–0.85\). The selected value, \($\alpha$ = 0.80\), therefore lies within a broad stable regime rather than representing an isolated sample-specific optimum. This supports the interpretation that Spartan is not dependent on a narrowly tuned value of \($\alpha\$), but instead operates robustly across a range of activation-enriched graph integration settings.

Conceptually, single-cell imaging-based spatial transcriptomics provides high spatial precision and relatively reliable local neighborhood structure. In this setting, local transcriptional deviations across spatial neighborhoods are expected to carry biologically meaningful information about anatomical boundaries, transitional zones, and fine-scale tissue organization. The LSA graph \(L\) is designed specifically to capture this neighborhood-conditioned activation structure. In contrast, the spatial graph \(S\) provides the physical neighborhood scaffold, while the gene-expression graph \(G\) provides expression-derived connectivity independent of physical adjacency.

We therefore assign a stronger effective contribution to \(L\) than to \(S\). This reflects the central design principle of Spartan: physical proximity alone is not the primary signal; rather, the key signal is how transcriptional activation varies across local spatial neighborhoods. The spatial graph defines where local relationships can occur, whereas the LSA graph determines how strongly those relationships are activated by local expression structure.

The fixed setting \($\alpha=0.80,\beta_1=0.10,\beta_2=0.40$\) therefore prioritizes activation-aware local structure while retaining both spatial topology and expression connectivity. Subsequent resolution analyses were performed under this fixed graph-weighting regime to evaluate clustering granularity without repeatedly changing the underlying graph integration model.

In [ ]:
# --- Demo parameters(VisiumHD)---
params = dict(
    spatial_coord="grid",              
    spatial_neighborhood="knn",       
    spatial_neighs=6,
    spatial_rings=2,
    total_pca_comps=50,
    pca_comps_extract=30,
    gene_neighs=15,
    alpha=0.80,
    beta1=0.10,
    beta2=0.40,
    resolution=0.6,#changed
    seed=1,
    key_added="spartan_domains", #additional option: copy = True/False (default:False) Scanpy feel
)

params

## 6. Run Spartan spatial domain detection

Spartan is applied to the Visium HD expression table using the parameters defined above. The function returns a copied `AnnData` object containing the constructed Spartan graphs and the predicted spatial domains. The resulting domain labels are then copied back into the `SpatialData` table so that they can be rendered on the original tissue geometry.

In [ ]:
# Run Spartan spatial domains(VisiumHD)
adata = sp.tl.spartan_spatial_domains(sdata['manual_analysis_count_area_filtered'], **params,copy=True) 

# Inspect domain counts
adata.obs[params["key_added"]].value_counts().head(5)

sdata['manual_analysis_count_area_filtered'].obs['spartan_domains']=adata.obs['spartan_domains'].copy()

### Inspect Spartan output object

This inspection confirms that the returned `AnnData` object contains the expected observations, variables, embeddings, graph matrices, and Spartan domain labels after running the spatial domain workflow.

In [ ]:
adata

### Confirm graph outputs

Spartan stores each graph component in `adata.obsp`. This cell checks that the spatial graph, row-normalized spatial weights, LSA graph, gene-expression graph, and final aggregated graph were successfully generated. These graph objects are important for reproducibility because they define the exact input to Leiden clustering.

In [ ]:
# Confirm outputs exist
required_obsp = [
    "spartan_spatial_graph",
    "spartan_spatial_weights",
    "spartan_lsa_graph",
    "spartan_gene_graph",
    "spartan_joint_graph",
]
missing = [k for k in required_obsp if k not in adata.obsp]
print("Missing:", missing)
for k in required_obsp:
    if k in adata.obsp:
        mat = adata.obsp[k]
        print(k, type(mat), getattr(mat, "shape", None))


## 7. Visualize Spartan spatial domains

The predicted Spartan domains are rendered directly on the Visium HD tissue geometry. This provides the primary spatial domain map used for biological interpretation and main-figure generation.

In [ ]:
#SpatialData plot for VisiumHD plotting
plt.rcParams['axes.grid'] = False 
sdata.pl.render_shapes(
    "analysis_geometry", color=
        "spartan_domains",
    method="matplotlib", table_name="manual_analysis_count_area_filtered"
).pl.show(coordinate_systems="RJ4_D1_Andy_Esophagus",figsize=(10, 10),title = "Spatial Domains",frameon=False, dpi=600)


## 8. Gastroesophageal junction domain and marker validation

Domain 5 is examined as the gastroesophageal junction (GEJ)-associated domain. The domain is plotted spatially and compared with the expression of `PLA2G2A`, a marker used here to support the biological interpretation of this region.

In [ ]:
#SpatialData plot for VisiumHD plotting
plt.rcParams['axes.grid'] = False 
sdata.pl.render_shapes(
    "analysis_geometry", color=
        "spartan_domains",groups=["5"],
    method="matplotlib", table_name="manual_analysis_count_area_filtered"
).pl.show(coordinate_systems="RJ4_D1_Andy_Esophagus",figsize=(10, 10),title = "Domain 5 (Gastroesophageal junction)",frameon=False, dpi=600)

#SpatialData plot for VisiumHD plotting
plt.rcParams['axes.grid'] = False 
sdata.pl.render_shapes(
    "analysis_geometry",
    cmap="Reds",
     color=
        "PLA2G2A",
    method="matplotlib", table_name="manual_analysis_count_area_filtered"
).pl.show(coordinate_systems="RJ4_D1_Andy_Esophagus",figsize=(10, 10),title = "PLA2G2A",frameon=False, dpi=600)


## 9. Subcluster the GEJ-associated domain

To further resolve heterogeneity within the GEJ-associated domain, the aggregated Spartan graph `J` is subset to spots assigned to domain 5. Leiden clustering is then re-run on this domain-specific subgraph. This allows the GEJ region to be decomposed into finer subdomains while preserving the graph structure learned by Spartan at the whole-tissue level.

In [ ]:
domain_name = '5'
domain_labels = adata.obs['spartan_domains']

domain_labels = domain_labels.to_numpy()

domain_5_indices = np.where(domain_labels==domain_name)[0]

domain5_JG = adata.obsp['spartan_joint_graph'][domain_5_indices,:][:,domain_5_indices]

jrgh = sp.tl.to_igraph(domain5_JG)

partition = leidenalg.find_partition(
        jrgh,
        leidenalg.RBConfigurationVertexPartition,
        weights=jrgh.es['weight'],n_iterations=-1,seed=seed,
        resolution_parameter= 0.2#resolution_parameter=1.1100000000000005
    
    )

adata.obs['spartan_domain5_clusters'] = "-1"

domain_5_labels = adata.obs.index[domain_5_indices]
adata.obs.loc[domain_5_labels, 'spartan_domain5_clusters'] = [str(c) for c in partition.membership]
sdata['manual_analysis_count_area_filtered'].obs['spartan_domain5_clusters'] = adata.obs['spartan_domain5_clusters'].astype("category")

### Inspect GEJ subdomain labels

This cell displays the newly assigned GEJ subdomain labels. Spots outside domain 5 are assigned `-1`, while spots inside domain 5 receive subcluster labels from the domain-specific Leiden run.

In [ ]:
sdata['manual_analysis_count_area_filtered'].obs['spartan_domain5_clusters'] 

### Visualize GEJ subdomains

The GEJ subclusters are rendered in tissue space. This visualization is used to inspect whether the GEJ domain contains spatially organized epithelial, mesenchymal, or transition-like subregions.

In [ ]:
#SpatialData plot for VisiumHD plotting
plt.rcParams['axes.grid'] = False 
sdata.pl.render_shapes(
    "analysis_geometry", color=
        "spartan_domain5_clusters",groups=["0","1","2","3","4"],
    method="matplotlib", table_name="manual_analysis_count_area_filtered"
).pl.show(coordinate_systems="RJ4_D1_Andy_Esophagus",figsize=(10, 10),title = "Sub Domains of Domain 5 (Gastroesophageal junction)",frameon=False, dpi=600)


## 10. Prepare raw-count-backed object for marker analysis

For marker-weight and signature analyses, the raw count layer is wrapped into an `AnnData` object and assigned to `tdata.raw`. This ensures that marker analyses can access raw expression values while retaining the same observations, variables, and metadata as the processed table.

In [ ]:
import scanpy as sc
from anndata import AnnData

# reference to your existing AnnData
tdata = sdata['manual_analysis_count_area_filtered']

# create a new AnnData with the raw counts
raw_adata = AnnData(
    X=tdata.layers['raw_counts'],
    obs=tdata.obs.copy(),
    var=tdata.var.copy(),
    uns=tdata.uns.copy()
)

# assign raw_adata to tdata.raw
tdata.raw = raw_adata

## 11. Pairwise marker-weight analysis: GEJ subdomain 0 versus subdomain 3

This analysis compares two GEJ subdomains using a logistic regression classifier based on selected epithelial and mesenchymal marker genes from the pairwise differential expression analysis (representative csv files are provided with the `Data` folder (please check `Pairwise DEG analysis for logistic regression of GEJ subdomains` folder inside the `Data` folder). The AUROC summarizes how well the marker set separates the two subdomains, while the learned coefficients indicate which genes contribute positively or negatively to the distinction.

This is an interpretability step: it is not used to define Spartan domains, but to characterize the biological programs enriched in the subdomains discovered by Spartan.

In [ ]:
pos_markers = ["CKB", "MT-CO3", "IGFBP2", "MUC5AC", "PLA2G2A", "MT-ND4L", "MUC1", "AGR2", "MUC5B"]  # Sub0 enriched (Epithelial)
neg_markers = ["COL1A1", "COL3A1", "COL1A2", "COL5A1", "COL6A2", "TPM2", "FBN1", "VIM"]  # Sub3 enriched (Mesenchymal)

# Filter to markers present in dataset
pos_markers = [g for g in pos_markers if g in sdata['manual_analysis_count_area_filtered'].var_names]
neg_markers = [g for g in neg_markers if g in sdata['manual_analysis_count_area_filtered'].var_names]

# Prepare the feature matrix
X_pos = sdata['manual_analysis_count_area_filtered'].raw[:, pos_markers].X
X_neg = sdata['manual_analysis_count_area_filtered'].raw[:, neg_markers].X

if hasattr(X_pos, "toarray"):
    X_pos = X_pos.toarray()
if hasattr(X_neg, "toarray"):
    X_neg = X_neg.toarray()

X = np.hstack([X_pos, X_neg])
y = (sdata['manual_analysis_count_area_filtered'].obs['spartan_domain5_clusters'].isin(['0'])).astype(int)
mask = sdata['manual_analysis_count_area_filtered'].obs['spartan_domain5_clusters'].isin(['0', '3'])

# Fit weighted logistic regression
clf = LogisticRegression(max_iter=1000)
clf.fit(X[mask], y[mask])

# Compute AUROC
y_pred = clf.predict_proba(X[mask])[:, 1]
auc = roc_auc_score(y[mask], y_pred)
print("Weighted logistic regression AUROC Sub0 vs Sub3:", auc)

# Show marker weights
marker_names = pos_markers + neg_markers
weights = pd.Series(clf.coef_.flatten(), index=marker_names).sort_values(ascending=False)
print("Marker weights for Sub0 vs Sub3:")
print(weights)

# Optional: plot weights as a horizontal bar chart
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
weights.plot(kind='barh')
plt.title("Sub0 vs Sub3 marker weights")
plt.xlabel("Weight")
plt.ylabel("Marker")
plt.tight_layout()
plt.savefig("Sub0_vs_Sub3_marker_weights.pdf", dpi=600)  # save as PDF
plt.show()

### Marker-weight and signature visualization

This cell converts the pairwise classifier result into a figure-ready visualization. Marker weights are shown alongside signature-level expression patterns to support interpretation of the GEJ subdomain contrast.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl  
from sklearn.metrics import roc_auc_score
plt.rcParams['axes.grid'] = False 

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.titlesize": 60,   # ≈ Arial 18
    "axes.labelsize": 54,
    "xtick.labelsize": 40,
    "ytick.labelsize": 40,
    "legend.fontsize": 54
})
# ------------------------
# 1. Define markers and weights for Sub0 vs Sub3

markers = [
    "MUC5B", "MUC5AC", "AGR2", "CKB", "PLA2G2A", "MUC1", "IGFBP2", "MT-CO3", "MT-ND4L", # Sub0 (Epithelial) markers
    "VIM","COL6A2","COL5A1", "COL3A1", "FBN1", "TPM2", "COL1A1", "COL1A2"             # Sub3 (Mesenchymal) markers
]

weights = np.array([
     1.890961,  1.659756,  1.290972,  1.008206,  0.999358,  0.988881,  0.890398,  0.423705,  0.196283, # Positive weights
     0.061735, -0.063662, -0.141832, -0.358949, -0.545108, -0.612986, -0.636676, -0.693169  # Negative weights
])

# Filter markers present in AnnData
markers_present = [g for g in markers if g in sdata['manual_analysis_count_area_filtered'].var_names]
weights_present = weights[[markers.index(g) for g in markers_present]]

# ------------------------
# 2. Compute Sub0 signature score
# ------------------------
X = sdata['manual_analysis_count_area_filtered'].raw[:, markers_present].X
if hasattr(X, "toarray"):
    X = X.toarray()

signature_score = X @ weights_present  # weighted sum

# ------------------------
# 3. AUROC calculation Sub0 vs Sub3
# ------------------------
cluster_obs = sdata['manual_analysis_count_area_filtered'].obs['spartan_domain5_clusters']
mask = cluster_obs.isin(['0', '3'])
y = (cluster_obs[mask] == '0').astype(int)  # Sub0 positive
auc = roc_auc_score(y, signature_score[mask])
print("Weighted logistic regression AUROC Sub0 vs Sub3:", auc)

# ------------------------
# 4. Create figure
# ------------------------
fig, axes = plt.subplots(1, 2, figsize=(48,13))

# Bar plot of weights
sns.barplot(
    x=weights_present,
    y=markers_present,
    palette=['#d73027' if w>0 else '#4575b4' for w in weights_present],
    ax=axes[0]
)
axes[0].set_xlabel("Marker weight")
axes[0].set_ylabel("Marker")
axes[0].set_title(f"Sub0 vs Sub3 Marker Weights\nAUROC = {auc:.3f}")

# Violin plot of signature score
italic_labels = [rf"$\it{{{g}}}$" for g in markers_present]
axes[0].set_yticklabels(italic_labels)
df_sig = pd.DataFrame({
    "spartan_domain5_clusters": cluster_obs[mask],
    "signature_score": signature_score[mask]
})
sns.violinplot(
    x="spartan_domain5_clusters",
    y="signature_score",
    data=df_sig,
    palette=["#4575b4","#d73027"],
    ax=axes[1]
)
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Weighted signature score")
axes[1].set_title("Sub0 vs Sub3 signature score distribution")

plt.tight_layout()
plt.savefig("Sub0_vs_Sub3_signature.pdf", dpi=600)  # save as PDF
plt.show()

## 12. Pairwise marker-weight analysis: GEJ subdomain 0 versus subdomain 1

This cell performs a second pairwise comparison between GEJ subdomains. The marker sets are chosen to contrast stromal/ECM-associated genes with epithelial or mucosal-associated genes. As above, the classifier is used only for post hoc biological interpretation of Spartan-defined subdomains.

In [ ]:
# Subset adata to Sub1 and Sub2 only
mask = sdata['manual_analysis_count_area_filtered'].obs['spartan_domain5_clusters'].isin(['0','1'])
adata_sub = sdata['manual_analysis_count_area_filtered'][mask]


pos_markers_sub1 = [
    "COL3A1",  
    "COL1A1",   
    "FN1",    
    "COL1A2",     
    "COL6A1",   
    "TNC",     
    "COL6A2",     
    "LTBP4",   
    "COL5A1"
]


neg_markers_sub1 = [
    "MUC5AC",     
    "CKB",  
    "MUC5B",  
    "MUC1",   
    "IGFBP2",  
    "SPINK1",  
    "CTSE",
    "FOXQ1"
]


# Keep only markers present in dataset
pos_markers_sub1 = [g for g in pos_markers_sub1 if g in adata_sub.var_names]
neg_markers_sub1 = [g for g in neg_markers_sub1 if g in adata_sub.var_names]

# Prepare feature matrix
X_pos = adata_sub.raw[:, pos_markers_sub1].X
X_neg = adata_sub.raw[:, neg_markers_sub1].X

if hasattr(X_pos, "toarray"): X_pos = X_pos.toarray()
if hasattr(X_neg, "toarray"): X_neg = X_neg.toarray()

X = np.hstack([X_pos, X_neg])
y = (adata_sub.obs['spartan_domain5_clusters'] == '1').astype(int)  # Sub2 = 1

# Fit logistic regression with weights
clf = LogisticRegression(solver='liblinear')
clf.fit(X, y)

# Get AUROC
y_pred = clf.predict_proba(X)[:,1]
auc = roc_auc_score(y, y_pred)
print("Weighted logistic regression AUROC Sub1 vs Sub0:", auc)

# Marker weights
weights = pd.Series(clf.coef_.ravel(), index=pos_markers_sub1 + neg_markers_sub1)
print("Marker weights for Sub1:\n", weights.sort_values(ascending=False))

### Marker weights for GEJ subdomain contrast

The marker weights from the subdomain comparison are formatted into a publication-style panel. Positive weights support one subdomain identity, while negative weights support the opposing subdomain identity.

In [ ]:
markers = [ "TNC","FN1","COL6A1","COL1A2","COL1A1","COL6A2","LTBP4","COL5A1","COL3A1",
            "IGFBP2",
            "CKB","FOXQ1","MUC1","MUC5B","MUC5AC","SPINK1","CTSE"
             
   ]

weights = np.array([
    1.094578,  
    1.013009,  
    0.606051,  
    0.478400,  
    0.315450,  
    0.218572,  
    0.211613,  
    0.140789,  
    0.114694,   
    -0.105566,
    -0.159003,
    -0.391329,
    -0.439077,
    -1.065195,
    -1.198447,
    -1.470830,
    -2.025923
    
])

# Filter markers actually present in AnnData
markers_present = [g for g in markers if g in sdata['manual_analysis_count_area_filtered'].var_names]
weights_present = weights[[markers.index(g) for g in markers_present]]

# ------------------------
# 2. Compute Sub2 signature score
# ------------------------
X = sdata['manual_analysis_count_area_filtered'].raw[:, markers_present].X
if hasattr(X, "toarray"):
    X = X.toarray()

signature_score = X @ weights_present  # weighted sum

# ------------------------
# 3. AUROC calculation Sub2 vs Sub1
# ------------------------
cluster_obs = sdata['manual_analysis_count_area_filtered'].obs['spartan_domain5_clusters']
mask = cluster_obs.isin(['0', '1'])
y = (cluster_obs[mask] == '1').astype(int)  # Sub2 positive
auc = roc_auc_score(y, signature_score[mask])
print("Weighted logistic regression AUROC Sub1 vs Sub0:", auc)

# ------------------------
# 4. Create figure
# ------------------------
fig, axes = plt.subplots(1, 2, figsize=(48,13))

# Bar plot of weights
sns.barplot(x=weights_present, y=markers_present, palette=['#d73027' if w>0 else '#4575b4' for w in weights_present], ax=axes[0])
axes[0].set_xlabel("Marker weight")
axes[0].set_ylabel("Marker")
axes[0].set_title(f"Sub1 vs Sub0 Marker Weights\nAUROC = {auc:.3f}")

# Violin plot of signature score
italic_labels = [rf"$\it{{{g}}}$" for g in markers_present]
axes[0].set_yticklabels(italic_labels)
df_sig = pd.DataFrame({
    "spartan_domain5_clusters": cluster_obs[mask],
    "signature_score": signature_score[mask]
})
sns.violinplot(x="spartan_domain5_clusters", y="signature_score", data=df_sig, palette=["#4575b4","#d73027"], ax=axes[1])
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Weighted signature score")
axes[1].set_title("Sub1 vs Sub0 signature score distribution")

plt.tight_layout()
plt.savefig("Sub1_vs_Sub0_signature.pdf", dpi=600)
plt.show()

## 13. Pairwise marker-weight analysis: GEJ subdomain 1 versus subdomain 2

This comparison examines another pair of GEJ subdomains and evaluates whether collagen, stromal, mitochondrial, or epithelial-associated markers distinguish the two regions. The goal is to characterize finer-scale structure within the GEJ domain.

In [ ]:
# Subset adata to Sub1 and Sub2 only
mask = sdata['manual_analysis_count_area_filtered'].obs['spartan_domain5_clusters'].isin(['1','2'])
adata_sub = sdata['manual_analysis_count_area_filtered'][mask]

pos_markers_sub2 = [
    "COL14A1",  
    "IGFBP5",   
    "FBLN1",    
    "DCN",      
    "IGFBP2",   
    "TPM2",     
    "FBN1",     
    "COL1A2",   
    "LTBP4",    
    "COL1A1"   
]

neg_markers_sub2 = [
    "LRP4",     
    "MT-ND4L",  
    "MT-CO3",   
    "MT-ND2",   
    "MT-CO2",   
    "MT-ATP6",  
    "MT-ND4"    
]


# Keep only markers present in dataset
pos_markers_sub2 = [g for g in pos_markers_sub2 if g in adata_sub.var_names]
neg_markers_sub2 = [g for g in neg_markers_sub2 if g in adata_sub.var_names]

# Prepare feature matrix
X_pos = adata_sub.raw[:, pos_markers_sub2].X
X_neg = adata_sub.raw[:, neg_markers_sub2].X

if hasattr(X_pos, "toarray"): X_pos = X_pos.toarray()
if hasattr(X_neg, "toarray"): X_neg = X_neg.toarray()

X = np.hstack([X_pos, X_neg])
y = (adata_sub.obs['spartan_domain5_clusters'] == '2').astype(int)  # Sub2 = 1

# Fit logistic regression with weights
clf = LogisticRegression(solver='liblinear')
clf.fit(X, y)

# Get AUROC
y_pred = clf.predict_proba(X)[:,1]
auc = roc_auc_score(y, y_pred)
print("Weighted logistic regression AUROC Sub2 vs Sub1:", auc)

Skip to Main
VisiumHDAnalysisSpartan
Last Checkpoint: 56 seconds ago
[Python 3 (ipykernel)]

# Marker weights
weights = pd.Series(clf.coef_.ravel(), index=pos_markers_sub2 + neg_markers_sub2)
print("Marker weights for Sub2:\n", weights.sort_values(ascending=False))

### Marker weights for subdomain 1 versus subdomain 2

This cell generates the corresponding marker-weight visualization for the subdomain 1 versus subdomain 2 contrast, supporting the subpanel-level interpretation of GEJ heterogeneity.

In [ ]:
markers = [
    "COL14A1", "IGFBP5", "FBLN1", "IGFBP2", "DCN", 
    "TPM2", "FBN1", "COL1A2", "LTBP4", "COL1A1", 
    "MT-ND4", "MT-ATP6", "MT-CO2", "MT-ND2", "MT-CO3", 
    "MT-ND4L", "LRP4"
]

weights = np.array([
    0.820603,  
    0.503057,  
    0.381627,  
    0.380707,  
    0.372632,  
    0.360801, 
    0.349021,  
    0.161862,  
    0.138276,  
    0.108188, 
    -0.119798, 
    -0.137694, 
    -0.205081, 
    -0.220221, 
    -0.266566, 
    -0.299005, 
    -1.315528  
])

# Filter markers actually present in AnnData
markers_present = [g for g in markers if g in sdata['manual_analysis_count_area_filtered'].var_names]
weights_present = weights[[markers.index(g) for g in markers_present]]

# ------------------------
# 2. Compute Sub2 signature score
# ------------------------
X = sdata['manual_analysis_count_area_filtered'].raw[:, markers_present].X
if hasattr(X, "toarray"):
    X = X.toarray()

signature_score = X @ weights_present  # weighted sum

# ------------------------
# 3. AUROC calculation Sub2 vs Sub1
# ------------------------
cluster_obs = sdata['manual_analysis_count_area_filtered'].obs['spartan_domain5_clusters']
mask = cluster_obs.isin(['1', '2'])
y = (cluster_obs[mask] == '2').astype(int)  # Sub2 positive
auc = roc_auc_score(y, signature_score[mask])
print("Weighted logistic regression AUROC Sub2 vs Sub1:", auc)

# ------------------------
# 4. Create figure
# ------------------------
fig, axes = plt.subplots(1, 2, figsize=(48,13))

# Bar plot of weights
sns.barplot(x=weights_present, y=markers_present, palette=['#d73027' if w>0 else '#4575b4' for w in weights_present], ax=axes[0])
axes[0].set_xlabel("Marker weight")
axes[0].set_ylabel("Marker")
axes[0].set_title(f"Sub2 vs Sub1 Marker Weights\nAUROC = {auc:.3f}")

# Violin plot of signature score
italic_labels = [rf"$\it{{{g}}}$" for g in markers_present]
axes[0].set_yticklabels(italic_labels)
df_sig = pd.DataFrame({
    "spartan_domain5_clusters": cluster_obs[mask],
    "signature_score": signature_score[mask]
})
sns.violinplot(x="spartan_domain5_clusters", y="signature_score", data=df_sig, palette=["#4575b4","#d73027"], ax=axes[1])
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Weighted signature score")
axes[1].set_title("Sub2 vs Sub1 signature score distribution")

plt.tight_layout()
plt.savefig("Sub2_vs_Sub1_signature.pdf", dpi=600)
plt.show()

## 14. Pairwise marker-weight analysis: GEJ subdomain 1 versus subdomain 4

This contrast evaluates whether subdomain 4 captures a distinct biological program relative to subdomain 1. The marker set includes genes associated with muscle, vascular, stromal, and epithelial programs, allowing the subdomain identity to be interpreted from learned marker coefficients.

In [ ]:
# ------------------------
# 1. Subset adata to Sub1 and Sub4 only
# ------------------------
mask = sdata['manual_analysis_count_area_filtered'].obs['spartan_domain5_clusters'].isin(['1','4'])
adata_sub = sdata['manual_analysis_count_area_filtered'][mask]


pos_markers_sub4 = [
    "TNNT3",    
    "CDH5",     
    "HBA2",     
    "GLI3",     
    "OSR2",     
    "FBLN1",    
    "COL6A2",   
    "ADIPOR1",  
    "ACTG2",    
    "DLC1"      
]

neg_markers_sub4 = [
    "PLA2G2A",  
    "MUC1",     
    "AGR2",    
    "MT-ND4L",  
    "MT-CO3",      
    "CKB",      
    "GSTP1",    
    "FOXQ1",    
    "CLDN7"     
]

# Keep only markers present in dataset
pos_markers_sub4 = [g for g in pos_markers_sub4 if g in adata_sub.var_names]
neg_markers_sub4 = [g for g in neg_markers_sub4 if g in adata_sub.var_names]

# ------------------------
# 3. Prepare feature matrix
# ------------------------
X_pos = adata_sub.raw[:, pos_markers_sub4].X
X_neg = adata_sub.raw[:, neg_markers_sub4].X

if hasattr(X_pos, "toarray"): X_pos = X_pos.toarray()
if hasattr(X_neg, "toarray"): X_neg = X_neg.toarray()

X = np.hstack([X_pos, X_neg])
y = (adata_sub.obs['spartan_domain5_clusters'] == '4').astype(int)  # Sub4 = 1

# ------------------------
# 4. Fit logistic regression with weights
# ------------------------
clf = LogisticRegression(solver='liblinear')
clf.fit(X, y)

# ------------------------
# 5. Compute AUROC
# ------------------------
y_pred = clf.predict_proba(X)[:,1]
auc = roc_auc_score(y, y_pred)
print("Weighted logistic regression AUROC Sub4 vs Sub1:", auc)

# ------------------------
# 6. Show marker weights
# ------------------------
weights = pd.Series(clf.coef_.ravel(), index=pos_markers_sub4 + neg_markers_sub4).sort_values(ascending=False)
print("Marker weights for Sub4 vs Sub1:\n", weights)

### Marker weights and signature scores for subdomain 4 versus subdomain 1

This cell formats the classifier-derived marker weights and signature scores into a main-figure-ready panel. The AUROC provides a compact measure of how separable the two subdomains are using the selected marker program.

In [ ]:
markers = [
    "TNNT3", "ADIPOR1", "OSR2", "HBA2", "GLI3", 
    "DLC1", "FBLN1", "CDH5", "ACTG2", "MT-CO3",
    "AGR2", "COL6A2", "MT-ND4L", "CKB", "GSTP1", 
    "MUC1", "CLDN7", "FOXQ1", "PLA2G2A"
]

weights = np.array([
    1.375275,  
    0.893857, 
    0.887244,  
    0.743800,  
    0.395342,  
    0.320044,  
    0.234845,  
    0.182721,  
    0.175988,  
    0.027143,  
    -0.033125, 
    -0.066873, 
    -0.082325, 
    -0.290088, 
    -0.322156, 
    -0.564874, 
    -0.710879, 
    -1.049227,
    -1.249147  
])

# Filter markers present in AnnData
markers_present = [g for g in markers if g in sdata['manual_analysis_count_area_filtered'].var_names]
weights_present = weights[[markers.index(g) for g in markers_present]]

# ------------------------
# 2. Compute Sub4 signature score
# ------------------------
mask = sdata['manual_analysis_count_area_filtered'].obs['spartan_domain5_clusters'].isin(['1','4'])
adata_sub = sdata['manual_analysis_count_area_filtered'][mask]

X = adata_sub.raw[:, markers_present].X
if hasattr(X, "toarray"):
    X = X.toarray()

signature_score = X @ weights_present  # weighted sum

# ------------------------
# 3. AUROC calculation Sub4 vs Sub1
# ------------------------
cluster_obs = adata_sub.obs['spartan_domain5_clusters']
y = (cluster_obs == '4').astype(int)  # Sub4 positive
auc = roc_auc_score(y, signature_score)
print("Weighted logistic regression AUROC Sub4 vs Sub1:", auc)

# ------------------------
# 4. Create figure
# ------------------------
fig, axes = plt.subplots(1, 2, figsize=(48,13))

# Bar plot of weights
sns.barplot(
    x=weights_present, 
    y=markers_present, 
    palette=['#d73027' if w>0 else '#4575b4' for w in weights_present], 
    ax=axes[0]
)
axes[0].set_xlabel("Marker weight")
axes[0].set_ylabel("Marker")
axes[0].set_title(f"Sub4 vs Sub1 Marker Weights\nAUROC = {auc:.3f}")

# Violin plot of signature score
italic_labels = [rf"$\it{{{g}}}$" for g in markers_present]
axes[0].set_yticklabels(italic_labels)
df_sig = pd.DataFrame({
    "spartan_domain5_clusters": cluster_obs,
    "signature_score": signature_score
})
sns.violinplot(
    x="spartan_domain5_clusters", 
    y="signature_score", 
    data=df_sig, 
    palette=["#4575b4","#d73027"], 
    ax=axes[1]
)
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Weighted signature score")
axes[1].set_title("Sub4 vs Sub1 signature score distribution")

plt.tight_layout()
plt.savefig("Sub4_vs_Sub1_signature.pdf", dpi=600)  # save as PDF
plt.show()


## 15. Domain 0: outer muscularis layer and TAGLN expression

Domain 0 is visualized as the outer muscularis-associated region. The expression of `TAGLN`, a smooth-muscle-associated marker, is plotted alongside the domain to support this anatomical interpretation.

In [ ]:
#SpatialData plot for VisiumHD plotting
plt.rcParams['axes.grid'] = False 
sdata.pl.render_shapes(
    "analysis_geometry", color=
        "spartan_domains",groups=["0"],
    method="matplotlib", table_name="manual_analysis_count_area_filtered"
).pl.show(coordinate_systems="RJ4_D1_Andy_Esophagus",figsize=(10, 10),title = "Domain 0 (outer muscularis layer)",frameon=False, dpi=600)

#SpatialData plot for VisiumHD plotting
plt.rcParams['axes.grid'] = False 
sdata.pl.render_shapes(
    "analysis_geometry",
    cmap="Reds",
     color=
        "TAGLN",
    method="matplotlib", table_name="manual_analysis_count_area_filtered"
).pl.show(coordinate_systems="RJ4_D1_Andy_Esophagus",figsize=(10, 10),title = "TAGLN",frameon=False, dpi=600)


## 16. Domain 3: mesenchymal stroma and FN1 expression

Domain 3 is examined as a mesenchymal stromal compartment. `FN1` expression is plotted as a representative stromal/ECM-associated gene supporting the biological identity of this domain.

In [ ]:
#SpatialData plot for VisiumHD plotting
plt.rcParams['axes.grid'] = False 
sdata.pl.render_shapes(
    "analysis_geometry", color=
        "spartan_domains",groups=["3"],
    method="matplotlib", table_name="manual_analysis_count_area_filtered"
).pl.show(coordinate_systems="RJ4_D1_Andy_Esophagus",figsize=(10, 10),title = "Domain 3 (mesenchymal stroma)",frameon=False, dpi=600)

#SpatialData plot for VisiumHD plotting
plt.rcParams['axes.grid'] = False 
sdata.pl.render_shapes(
    "analysis_geometry",
    cmap="Reds",
     color=
        "FN1",
    method="matplotlib", table_name="manual_analysis_count_area_filtered"
).pl.show(coordinate_systems="RJ4_D1_Andy_Esophagus",figsize=(10, 10),title = "FN1",frameon=False, dpi=600)

## 17. Domain 4: luminal epithelium and KRT5 expression

Domain 4 is visualized together with `KRT5` expression to support interpretation of this epithelial compartment. This domain-level marker analysis provides a direct link between Spartan spatial partitions and known tissue-associated gene programs.

In [ ]:
#SpatialData plot for VisiumHD plotting
plt.rcParams['axes.grid'] = False 
sdata.pl.render_shapes(
    "analysis_geometry", color=
        "spartan_domains",groups=["4"],
    method="matplotlib", table_name="manual_analysis_count_area_filtered"
).pl.show(coordinate_systems="RJ4_D1_Andy_Esophagus",figsize=(10, 10),title = "Domain 4 (luminal epithelium)",frameon=False, dpi=600)

#SpatialData plot for VisiumHD plotting
plt.rcParams['axes.grid'] = False 
sdata.pl.render_shapes(
    "analysis_geometry",
    cmap="Reds",
     color=
        "KRT5",
    method="matplotlib", table_name="manual_analysis_count_area_filtered"
).pl.show(coordinate_systems="RJ4_D1_Andy_Esophagus",figsize=(10, 10),title = "KRT5",frameon=False, dpi=600)


## 18. Domain 2: gastric mucosal epithelium and SPINK1 expression

Domain 2 is inspected as a gastric mucosal epithelial region. `SPINK1` expression is shown as a representative marker supporting the biological identity of this domain.

In [ ]:
#SpatialData plot for VisiumHD plotting
plt.rcParams['axes.grid'] = False 
sdata.pl.render_shapes(
    "analysis_geometry", color=
        "spartan_domains",groups=["2"],
    method="matplotlib", table_name="manual_analysis_count_area_filtered"
).pl.show(coordinate_systems="RJ4_D1_Andy_Esophagus",figsize=(10, 10),title = "Domain 2 (gastric mucosal epithelium)",frameon=False, dpi=600)

#SpatialData plot for VisiumHD plotting
plt.rcParams['axes.grid'] = False 
sdata.pl.render_shapes(
    "analysis_geometry",
    cmap="Reds",
     color=
        "SPINK1",
    method="matplotlib", table_name="manual_analysis_count_area_filtered"
).pl.show(coordinate_systems="RJ4_D1_Andy_Esophagus",figsize=(10, 10),title = "SPINK1",frameon=False, dpi=600)


### Re-inspect the table before expression summaries

Before generating expression summaries, the main `AnnData` table is inspected again to confirm that Spartan domains, GEJ subdomains, and expression layers are available.

In [ ]:
sdata['manual_analysis_count_area_filtered']

## 19. Mean marker expression across Spartan domains

This cell summarizes the mean expression of selected representative genes across Spartan domains. These genes correspond to major tissue compartments or boundary-associated regions, including the GEJ, muscularis, stroma, and gastric mucosa. The resulting bar plots provide a compact domain-level marker validation panel.

In [ ]:
import math

genes = ["PLA2G2A","TAGLN", "FN1", "KRT5", "SPINK1"]
#sdata['manual_analysis_count_area_filtered']
sdata['manual_analysis_count_area_filtered'].layers['log1pX'] = sdata['manual_analysis_count_area_filtered'].X
domain_key = "spartan_domains"
# Calculate grid size (2 columns, dynamic rows)
cols = 2
rows = math.ceil(len(genes) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 5))
axes = axes.flatten() # Flatten to iterate easily

for i, gene in enumerate(genes):
    df = pd.DataFrame({
      "domain": sdata['manual_analysis_count_area_filtered'].obs[domain_key],
      "expr": sdata['manual_analysis_count_area_filtered'][:, gene].layers['log1pX'].toarray().flatten()
      })

    mean_expr = df.groupby("domain").mean().sort_values("expr")
    
    # Plot on the specific subplot axis
    mean_expr.expr.plot(kind='bar', color='tab:green', ax=axes[i], rot=45)
    
    gene_it = rf"$\it{{{gene}}}$"
    axes[i].set_title(f"Mean {gene_it} across domains")
    axes[i].set_ylabel("Expression")

# Remove any unused subplots (if genes aren't a multiple of 2)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig("combined_genes_expression.pdf", dpi=600)
plt.show()

### Distribution of SPINK1 expression across domains

The violin plot shows the distribution of `SPINK1` expression across Spartan domains. This complements the spatial plot by quantifying whether the marker is selectively enriched in the expected domain.

In [ ]:
plt.figure(figsize=(18, 12)) 


sc.pl.violin(
    sdata['manual_analysis_count_area_filtered'],
    'SPINK1',
    layer='log1pX',
    groupby='spartan_domains',
    xlabel='Domain',
    density_norm='count',
    use_raw=False,
    #figsize=(14, 9),
    rotation =45,
    stripplot=True,
    show=False
)

plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.xlabel("Domain", fontsize=18)
plt.ylabel("SPINK1", fontsize=18)

#plt.savefig("Domain2S1.pdf", dpi=600, bbox_inches="tight")
plt.show()

## 20. Functional module scoring across stromal domains

This section evaluates whether Spartan separates functionally distinct stromal regions. Three gene modules are defined: structural ECM, glycosaminoglycan/sulfation-related genes, and vascular/stromal signaling genes. Module scores are computed per spot and optionally spatially smoothed using local KNN neighborhoods.

These module scores are used to compare stromal domains and support the interpretation that Spartan identifies functionally distinct stromal microenvironments rather than only broad anatomical regions.

In [ ]:
import numpy as np
import scanpy as sc
from sklearn.neighbors import NearestNeighbors
sdata['manual_analysis_count_area_filtered'].obsp['connectivities']=adata.obsp["spartan_spatial_graph"]

ECM_structural = [
    "COL1A1", "COL3A1", "COL5A1", "COL6A1",
    "DCN", "LUM", "FN1", "SPARC"
]

GAG_sulfation = [
    "DCN", "VCAN", "BGN", "DSEL", "CSPG4", "CHSY1"
]

Vascular_stroma = [
    "PDGFRB", "ROBO1", "ROBO2", "BMP4",
    "TGFB2", "NOTCH1", "TBX2", "EFEMP2"
]


# -----------------------------
# Helpers
# -----------------------------
def _present_genes(adata, genes):
    """Return genes present in adata.var_names; also print what's missing."""
    genes = list(dict.fromkeys(genes))  # de-dup, keep order
    present = [g for g in genes if g in adata.var_names]
    missing = [g for g in genes if g not in adata.var_names]
    if missing:
        print(f"[WARN] Missing {len(missing)} genes (ignored): {missing}")
    if len(present) < 3:
        print(f"[WARN] Only {len(present)} genes left after filtering. Consider expanding the list.")
    return present


def score_module(adata, gene_list, score_name, layer="log1pX"):
    """
    Robust module scoring:
      - filters missing genes
      - uses provided layer if available, otherwise falls back to X
      - avoids use_raw surprises
    """
    genes = _present_genes(adata, gene_list)

    # If requested layer doesn't exist, fall back gracefully
    layer_to_use = layer if (layer is not None and layer in adata.layers.keys()) else None
    if layer is not None and layer_to_use is None:
        print(f"[WARN] layer='{layer}' not found in adata.layers; using adata.X instead.")

    sc.tl.score_genes(
        adata,
        gene_list=genes,
        score_name=score_name,
        use_raw=False,     
        layer=layer_to_use 
    )

def spatial_smooth_knn(
    adata,
    score_key,
    k=8,
    new_key=None,
    include_self=False
):
    """
    Simple spatial KNN smoothing in coordinate space.
    Fixes common issues:
      - excludes self by default (prevents "shrinking" toward the original value)
      - uses k+1 neighbors when excluding self
    """
    coords = adata.obsm["spatial"]
    scores = np.asarray(adata.obs[score_key].values, dtype=float)

    if new_key is None:
        new_key = f"{score_key}_spatial"

    n = coords.shape[0]
    k_eff = min(k + (0 if include_self else 1), n)

    nbrs = NearestNeighbors(n_neighbors=k_eff, algorithm="auto").fit(coords)
    _, indices = nbrs.kneighbors(coords)

    if not include_self:
        # first neighbor is the point itself (distance 0)
        indices = indices[:, 1:]
    else:
        indices = indices[:, :k]

    smoothed = scores[indices].mean(axis=1)
    adata.obs[new_key] = smoothed
    return new_key

    
adata1.layers['log1pX'] = sdata["manual_analysis_count_area_filtered"].X 


score_module(adata1, ECM_structural, "ECM_structural_score", layer="log1pX")
score_module(adata1, GAG_sulfation,  "GAG_sulfation_score",  layer="log1pX")
score_module(adata1, Vascular_stroma,"Vascular_stroma_score",layer="log1pX")

ecm_k = spatial_smooth_knn(adata1, "ECM_structural_score", k=8, include_self=False)
gag_k = spatial_smooth_knn(adata1, "GAG_sulfation_score",  k=8, include_self=False)
vas_k = spatial_smooth_knn(adata1, "Vascular_stroma_score",k=8, include_self=False)

group_medians = (
    sdata['manual_analysis_count_area_filtered'].obs
    .groupby("spartan_domains")[
        ["ECM_structural_score_spatial",
         "GAG_sulfation_score_spatial",
         "Vascular_stroma_score_spatial"]
    ]
    .median()
)

print(group_medians)

## 21. Compare stromal module scores across domains 1, 6, and 7

The median module scores for domains 1, 6, and 7 are plotted as lollipop charts. This provides a compact visual summary of how the stromal domains differ across ECM, glycosaminoglycan/sulfation, and vascular/stromal programs.

In [ ]:
TITLE_FZ = 60
LABEL_FZ = 54
TICK_FZ  = 44

domains_of_interest = ["1", "6", "7"]
group_medians_sub = group_medians.loc[domains_of_interest]
domains = group_medians_sub.index
scores = group_medians_sub.columns
colors = ['skyblue', 'lightgreen', 'salmon']  

fig, axes = plt.subplots(1, 3, figsize=(34, 10), sharey=True)

for ax, score, color in zip(axes, scores, colors):
    vals = group_medians_sub[score]

    ax.hlines(domains, 0, vals, color=color, linewidth=8)
    ax.scatter(vals, domains, color=color, s=200,alpha=1)

    ax.set_title(score.replace("_", " "), fontsize=TITLE_FZ, pad=25)
    ax.set_xlabel("Median score", fontsize=LABEL_FZ)

    ax.tick_params(axis="x", labelsize=TICK_FZ)
    ax.tick_params(axis="y", labelsize=TICK_FZ)

    ax.set_xlim(vals.min() * 0.9, vals.max() * 1.1)

    ax.grid(True, axis="x", linewidth=1, alpha=0.4) 

axes[0].set_ylabel("Spatial domain", fontsize=LABEL_FZ)

plt.tight_layout()

## 22. Pathway-level functional blueprint of gastric stroma

This heatmap summarizes pathway-level normalized enrichment scores across stromal domains (Pathway scores are obtianed from standard GSEA analysis, GSEA analysis results are provide with the `Data` folder. `Please check GSEA outputs for Domains 1, 6, 7` folder). It provides a higher-level functional interpretation of the stromal spatial domains and supports the Figure 6 narrative that Spartan resolves biologically distinct stromal programs within the developing gastrointestinal tissue.

In [ ]:
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.titlesize": 60,   # ≈ Arial 18
    "axes.labelsize": 54,
    "xtick.labelsize": 40,
    "ytick.labelsize": 40,
    "legend.fontsize": 54
})
# -------------------------
# 1) Data
# -------------------------
data = {
    "Pathways": [
        "Collagen fibril organization (GO:0030199)",
        "Extracellular structure organization (GO:0043062)",
        "Dermatan sulfate metabolic process (GO:0030205)",
        "Negative regulation of immune system (GO:0002683)",
        "Aorta development (GO:0035904)",
        "Regulation of immune response (GO:0050776)",
        "Protein digestion and absorption (hsa04974)"
    ],
    "Domain 1": [2.12, 2.26, 1.84, 1.61, 1.86, 1.96, 2.04],
    "Domain 6": [1.94, 2.00, 2.04, 2.07, 1.95, 1.85, 2.28],
    "Domain 7": [1.79, 1.95, 1.81, 1.68, 2.08, 2.11, 1.97]
}

df_plot = pd.DataFrame(data).set_index("Pathways")


TITLE_FZ = 60
XTICK_FZ = 50
YTICK_FZ = 50
ANNOT_FZ = 54
CBAR_LABEL_FZ = 54
CBAR_TICK_FZ = 44

# -------------------------
# 3) Plot
# -------------------------
plt.figure(figsize=(34, 18))

ax = sns.heatmap(
    df_plot,
    #cmap="Spectral",
    #center=0,
    annot=True,
    fmt=".2f",  
    annot_kws={"size": ANNOT_FZ},  
    linewidths=1.5,
    linecolor="white",
    cbar_kws={"label": "NES Score"}
)

# Title
ax.set_title("Spatial Functional Blueprint of Gastric Stroma", fontsize=TITLE_FZ, pad=20)

# Axis labels 
ax.set_ylabel("")
ax.set_xlabel("")

# Tick label font sizes
ax.tick_params(axis="x", labelsize=XTICK_FZ)
ax.tick_params(axis="y", labelsize=YTICK_FZ)

# Colorbar font sizes
cbar = ax.collections[0].colorbar
cbar.set_label("NES Score", fontsize=CBAR_LABEL_FZ)
cbar.ax.tick_params(labelsize=CBAR_TICK_FZ)

plt.grid(False)
plt.tight_layout()
plt.show()


## 23. Summary

This notebook demonstrates how Spartan can be used to analyze high-resolution Visium HD data in a `SpatialData` framework. The workflow identifies major tissue domains, resolves a GEJ-associated domain into finer subdomains, validates domain identities using representative marker genes, and compares functional stromal programs across selected domains.

The key outputs support the main-paper Figure 5 and Figure 6 panels by showing that Spartan can recover anatomically meaningful high-resolution domains and reveal biologically interpretable substructure in the developing human esophagus and stomach sample.